# Problem Transformation Methods - Starter Notebook
This notebook compares Binary Relevance and Label Powerset as two classic transformations for multi-label tasks.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_multilabel_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier

In [ ]:
X, Y = make_multilabel_classification(n_samples=900, n_features=16, n_classes=5, n_labels=2, random_state=42)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=42)

# Binary relevance
br_model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
br_model.fit(X_train, Y_train)
br_pred = br_model.predict(X_test)

# Label powerset transformation
def to_labelset(y_row):
    indices = np.where(y_row == 1)[0]
    return '|'.join(map(str, indices.tolist())) if len(indices) else 'none'

lp_train = np.array([to_labelset(row) for row in Y_train])
lp_test = np.array([to_labelset(row) for row in Y_test])
lp_model = LogisticRegression(max_iter=1000)
lp_model.fit(X_train, lp_train)
lp_pred_strings = lp_model.predict(X_test)

In [ ]:
def from_labelset(label, n_classes):
    vec = np.zeros(n_classes, dtype=int)
    if label != 'none':
        for token in label.split('|'):
            vec[int(token)] = 1
    return vec

lp_pred = np.vstack([from_labelset(label, Y.shape[1]) for label in lp_pred_strings])

results = pd.DataFrame([
    {'Method': 'Binary Relevance', 'Micro F1': f1_score(Y_test, br_pred, average='micro')},
    {'Method': 'Label Powerset', 'Micro F1': f1_score(Y_test, lp_pred, average='micro')},
]).set_index('Method').round(3)
results